**Final Capstone Project**

**Part 1 - Build the governed dataset**

In [1]:
import pandas as pd, numpy as np
DATA = "." # folder holding your capstone CSV files
s = pd.read_csv(f"{DATA}/capstone_subscribers.csv")
print("rows in file :", len(s))
print("distinct customer_id :", s.customer_id.nunique())
print("duplicate customer_id :", s.duplicated(subset=["customer_id"]).sum())
print("distinct region values:", s.region.nunique())
print("missing avg_monthly_gb:", s.avg_monthly_gb.isna().sum())
print("missing days_since_last_recharge:", s.days_since_last_recharge.isna().sum())

rows in file : 3222
distinct customer_id : 3200
duplicate customer_id : 22
distinct region values: 18
missing avg_monthly_gb: 99
missing days_since_last_recharge: 58


In [2]:
# 1) text standardisation - do this BEFORE any groupby
s["region"] = s.region.str.strip().str.title()
print("region values after cleaning:", s.region.nunique(), "->", sorted(s.region.unique()))
# 2) de-duplicate on the business key, keeping the first occurrence
before = len(s)
s = s.drop_duplicates(subset=["customer_id"], keep="first").copy()
print(f"removed {before - len(s)} duplicate subscriber rows, {len(s)} remain")
# 3) fill numeric gaps with a median and RECORD that you did it
for c in ["avg_monthly_gb", "days_since_last_recharge"]:
 s[c + "_was_missing"] = s[c].isna().astype(int)
 s[c] = s[c].fillna(s[c].median())
print("rows flagged as imputed:", s[["avg_monthly_gb_was_missing",
 "days_since_last_recharge_was_missing"]].sum().to_dict())


region values after cleaning: 6 -> ['Bengaluru', 'Chennai', 'Delhi', 'Hyderabad', 'Kolkata', 'Mumbai']
removed 22 duplicate subscriber rows, 3200 remain
rows flagged as imputed: {'avg_monthly_gb_was_missing': 96, 'days_since_last_recharge_was_missing': 58}


In [3]:
OUT_PATH = "capstone_integrated.csv"
def run_pipeline():
 df = pd.read_csv(f"{DATA}/capstone_subscribers.csv")
 df["region"] = df.region.str.strip().str.title()
 df = df.drop_duplicates(subset=["customer_id"], keep="first")
 # ... the rest of your transformations, and your track-specific joins ...
 df.to_csv(OUT_PATH, index=False)
 return len(df)
first = run_pipeline()
second = run_pipeline()
print(f"first run: {first} rows second run: {second} rows")
assert first == second, f"pipeline is not idempotent: {first} -> {second}"
print("pipeline is idempotent")

first run: 3200 rows second run: 3200 rows
pipeline is idempotent


**PART 2**

Track A - Churn prediction and retention strategy

In [4]:
u = pd.read_csv(f"{DATA}/capstone_usage_monthly.csv")
print("rows in file:", len(u), " duplicate (customer_id, month):",
 u.duplicated(subset=["customer_id","month"]).sum())
u = u.drop_duplicates(subset=["customer_id","month"], keep="first").copy()
print("after de-duplication:", len(u), "rows across", u.month.nunique(), "months")
agg = u.groupby("customer_id").agg(
 months_seen = ("month", "nunique"),
 tot_gb = ("data_gb", "sum"),
 tot_voice = ("voice_min", "sum"),
 tot_intl = ("intl_min", "sum"),
 tot_rev = ("revenue_inr", "sum"),
 zero_rev_months = ("revenue_inr", lambda x: int((x == 0).sum())),
 fails = ("failed_payment_count", "sum"),
).reset_index()
# the trend feature: recent three months against the first three
late = u[u.month >= "2026-05"].groupby("customer_id").data_gb.mean().rename("gb_late")
early = u[u.month < "2026-05"].groupby("customer_id").data_gb.mean().rename("gb_early")
agg = agg.merge(late, on="customer_id").merge(early, on="customer_id")
agg["gb_trend"] = agg.gb_late / agg.gb_early.clip(lower=0.01)
df = s.merge(agg, on="customer_id", how="left")
df["complaint_intensity"] = df.complaints_6m / (df.tenure_months / 6).clip(lower=1)
print("merged frame:", df.shape)

rows in file: 19238  duplicate (customer_id, month): 38
after de-duplication: 19200 rows across 6 months
merged frame: (3200, 34)


In [5]:
# evidence 1 - the offer column is a consequence, not a cause
print(df.groupby("churn").retention_offer_sent.mean().round(3))
# evidence 2 - total_charges is arpu multiplied by tenure
implied = df.total_charges / df.tenure_months.clip(lower=1)
print("correlation of implied ARPU with arpu:", round(implied.corr(df.arpu), 4))

churn
0    0.051
1    0.843
Name: retention_offer_sent, dtype: float64
correlation of implied ARPU with arpu: 0.9995


Measure what the leak is worth

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
def build_and_score(drop_cols, model=None):
 X = df.drop(columns=["churn", "customer_id"] + drop_cols)
 y = df["churn"]
 num = X.select_dtypes(include=np.number).columns.tolist()
 cat = X.select_dtypes(exclude=np.number).columns.tolist()
 pre = ColumnTransformer([
 ("n", Pipeline([("i", SimpleImputer(strategy="median")),
 ("s", StandardScaler())]), num),
 ("c", Pipeline([("i", SimpleImputer(strategy="most_frequent")),
 ("o", OneHotEncoder(handle_unknown="ignore"))]), cat)])
 Xtr, Xte, ytr, yte = train_test_split(
 X, y, test_size=0.25, stratify=y, random_state=42)
 clf = model if model is not None else RandomForestClassifier(
 n_estimators=300, min_samples_leaf=3, random_state=42)
 pipe = Pipeline([("pre", pre), ("clf", clf)]).fit(Xtr, ytr)
 proba = pipe.predict_proba(Xte)[:, 1]
 return roc_auc_score(yte, proba), proba, yte, pre
LEAKY = ["retention_offer_sent", "total_charges"]
auc_leak, _, _, _ = build_and_score([])
auc_clean, proba, yte, pre = build_and_score(LEAKY)
print(f"ROC-AUC with the leak : {auc_leak:.3f}")
print(f"ROC-AUC without it : {auc_clean:.3f}")
print(f"the leak was worth : {auc_leak - auc_clean:.3f}")

ROC-AUC with the leak : 0.948
ROC-AUC without it : 0.725
the leak was worth : 0.223


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score, recall_score, accuracy_score
models = {
 "Logistic regression": LogisticRegression(max_iter=2000),
 "Decision tree": DecisionTreeClassifier(max_depth=6, random_state=42),
 "Random forest": RandomForestClassifier(n_estimators=300,
 min_samples_leaf=3, random_state=42),
}
rows = []
for name, m in models.items():
 a, pr, yt, _ = build_and_score(LEAKY, m)
 p05 = (pr >= 0.5).astype(int)
 rows.append({"model": name, "roc_auc": round(a, 3),
 "precision": round(precision_score(yt, p05, zero_division=0), 3),
 "recall": round(recall_score(yt, p05), 3),
 "accuracy": round(accuracy_score(yt, p05), 3)})
pd.DataFrame(rows)


,model,roc_auc,precision,recall,accuracy
0,Logistic regression,0.741,0.708,0.240,0.794
1,Decision tree,0.649,0.500,0.188,0.760
2,Random forest,0.725,0.684,0.068,0.769


Compare against the do-nothing baseline

In [9]:
baseline = 1 - yte.mean()
print("predict nobody churns - accuracy :", round(baseline, 3))
print("random forest at 0.5 - accuracy :",
 round(((proba >= 0.5).astype(int) == yte).mean(), 3))

predict nobody churns - accuracy : 0.76
random forest at 0.5 - accuracy : 0.769


Choose a threshold from the cost of being wrong

In [10]:
from sklearn.metrics import confusion_matrix
COST_FN, COST_FP = 5880, 300
rows = []
for t in np.arange(0.05, 0.95, 0.05):
 pred = (proba >= t).astype(int)
 tn, fp, fn, tp = confusion_matrix(yte, pred).ravel()
 rows.append({"threshold": round(t, 2),
 "precision": round(precision_score(yte, pred, zero_division=0), 3),
 "recall": round(recall_score(yte, pred), 3),
 "flagged": int(pred.sum()),
 "expected_cost": fn * COST_FN + fp * COST_FP})
sweep = pd.DataFrame(rows)
sweep.sort_values("expected_cost").head(4)

,threshold,precision,recall,flagged,expected_cost
1,0.10,0.256,0.995,747,172680
0,0.05,0.240,0.995,796,187380
2,0.15,0.279,0.932,642,215340
3,0.20,0.316,0.812,494,313080


In [11]:
# the closed-form optimum, for comparison with your sweep
print("theoretical optimum:", round(COST_FP / (COST_FP + COST_FN), 4))

theoretical optimum: 0.0485


In [12]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
feats = ["avg_monthly_gb", "avg_voice_min", "tenure_months",
 "arpu", "complaints_6m", "days_since_last_recharge"]
X = SimpleImputer(strategy="median").fit_transform(df[feats])
X = StandardScaler().fit_transform(X) # never skip this
for kk in range(2, 9):
 km = KMeans(n_clusters=kk, n_init=10, random_state=42).fit(X)
 sil = silhouette_score(X, km.labels_, sample_size=2000, random_state=1)
 print(f"k={kk} inertia={km.inertia_:9.1f} silhouette={sil:.3f}")


k=2 inertia=  15575.2 silhouette=0.238
k=3 inertia=  13853.9 silhouette=0.235
k=4 inertia=  12290.8 silhouette=0.204
k=5 inertia=  10913.5 silhouette=0.212
k=6 inertia=   9710.8 silhouette=0.205
k=7 inertia=   9044.2 silhouette=0.176
k=8 inertia=   8674.9 silhouette=0.158


In [13]:
K = 3 # your choice - change it and justify it
km = KMeans(n_clusters=K, n_init=10, random_state=42).fit(X)
df["segment"] = km.labels_
profile = df.groupby("segment").agg(
 size = ("customer_id", "count"),
 mean_arpu = ("arpu", "mean"),
 mean_tenure = ("tenure_months", "mean"),
 mean_gb = ("avg_monthly_gb", "mean"),
 mean_complaints = ("complaints_6m", "mean"),
 churn_rate = ("churn", "mean")).round(2)
profile["share_pct"] = (profile["size"] / len(df) * 100).round(1)
profile

,size,mean_arpu,mean_tenure,mean_gb,mean_complaints,churn_rate,share_pct
segment,,,,,,,
0,300,242.52,25.10,7.57,3.86,0.44,9.4
1,2112,208.15,22.88,5.66,0.54,0.23,66.0
2,788,396.90,24.04,17.49,0.67,0.20,24.6


In [15]:
from sklearn.ensemble import IsolationForest
u["intl_share"] = u.intl_min / (u.voice_min + u.intl_min + 1)
u["revenue_per_gb"] = u.revenue_inr / u.data_gb.clip(lower=0.01)
u["zero_revenue"] = (u.revenue_inr == 0).astype(int)
u["data_to_voice"] = u.data_gb / u.voice_min.clip(lower=1)
feat = ["data_gb","voice_min","intl_min","sms_count","revenue_inr",
 "failed_payment_count","intl_share","revenue_per_gb","data_to_voice"]
Xa = StandardScaler().fit_transform(
 u[feat].replace([np.inf, -np.inf], 0).fillna(0))
for c in [0.005, 0.01, 0.02]:
 iso = IsolationForest(contamination=c, n_estimators=300,
 random_state=42).fit(Xa)
 u[f"flag_{c}"] = (iso.predict(Xa) == -1).astype(int)
 u[f"score_{c}"] = -iso.score_samples(Xa)
 n = int(u[f"flag_{c}"].sum())
 print(f"contamination {c:<6} flagged {n:>4} rows"
 f" ~{n/26:.1f}/week cost Rs {n*450:,}")

contamination 0.005  flagged   96 rows ~3.7/week cost Rs 43,200
contamination 0.01   flagged  192 rows ~7.4/week cost Rs 86,400
contamination 0.02   flagged  384 rows ~14.8/week cost Rs 172,800


Rank on risk and value together

In [17]:

# score every subscriber, not just the held-out fold
X_all = df.drop(columns=["churn", "customer_id"] + LEAKY)
full = Pipeline([("pre", pre), ("clf", RandomForestClassifier(
 n_estimators=300, min_samples_leaf=3, random_state=42))])
full.fit(X_all, df["churn"]) # refit on everything, for scoring only
df["risk"] = full.predict_proba(X_all)[:, 1]
df["expected_loss"] = df.risk * df.arpu * 12 # a year of revenue at risk
call_list = df.sort_values("expected_loss", ascending=False)
print(call_list[["customer_id","segment","risk","arpu","expected_loss"]].head(10))
# how different is this from ranking on risk alone?
top_risk = set(df.nlargest(300, "risk").customer_id)
top_value = set(df.nlargest(300, "expected_loss").customer_id)
print("overlap between the two top-300 lists:", len(top_risk & top_value))

     customer_id  segment      risk    arpu  expected_loss
2747   SUB101910        2  0.640085  660.72    5075.000408
1235   SUB101691        2  0.593025  528.97    3764.309609
1815   SUB102265        2  0.597669  518.01    3715.184050
2285   SUB102123        2  0.530976  560.99    3574.469118
1338   SUB100580        2  0.580846  504.33    3515.253995
1538   SUB102827        2  0.542684  535.53    3487.483267
584    SUB102057        2  0.696574  408.49    3414.523257
1654   SUB100356        2  0.642798  440.98    3401.535354
3153   SUB101952        0  0.650639  435.45    3399.851251
693    SUB102863        2  0.586294  475.08    3342.439538
overlap between the two top-300 lists: 131


In [18]:
# Part 3 - costed comparison for the target segment
seg0 = df[df.segment == 0]

segment_size = len(seg0)
base_churn   = seg0['churn'].mean()      # segment's historical churn rate
value_lost   = 5880                       # lifetime value of a lost subscriber (Rs)

print(f"Target segment: Segment 0 - High risk")
print(f"Subscribers in segment: {segment_size}")
print(f"Base churn rate (segment): {base_churn:.1%}")
print(f"Lifetime value per lost subscriber: Rs {value_lost:,}\n")

options = [
    {"name": "Do nothing",         "cost_per_sub": 0,   "reduction": 0.00},
    {"name": "Retention call",     "cost_per_sub": 300, "reduction": 0.15},
    {"name": "Bill credit + call", "cost_per_sub": 550, "reduction": 0.25},
]

print(f"{'Option':<20}{'Saved (subs)':>14}{'Spend (Rs)':>16}{'Net value (Rs)':>18}{'ROI':>8}")
print("-"*76)
results = []
for o in options:
    saved = segment_size * base_churn * o["reduction"]
    spend = segment_size * o["cost_per_sub"]
    net   = saved * value_lost - spend
    roi   = (net / spend) if spend > 0 else float('nan')
    results.append({**o, "saved": saved, "spend": spend, "net": net, "roi": roi})
    roi_str = f"{roi:.1f}x" if spend > 0 else "-"
    print(f"{o['name']:<20}{saved:>14.1f}{spend:>16,.0f}{net:>18,.0f}{roi_str:>8}")

best = max(results, key=lambda r: r["net"])
print(f"\nBest option by net value: {best['name']} (net Rs {best['net']:,.0f})")

print("\n--- Stress test: halve the most important assumption (reduction %) ---")
for o in options:
    if o["cost_per_sub"] == 0: continue
    half_reduction = o["reduction"] / 2
    saved = segment_size * base_churn * half_reduction
    spend = segment_size * o["cost_per_sub"]
    net   = saved * value_lost - spend
    print(f"{o['name']:<20} at half effectiveness ({half_reduction:.1%} reduction): net Rs {net:,.0f}")

Target segment: Segment 0 - High risk
Subscribers in segment: 300
Base churn rate (segment): 44.0%
Lifetime value per lost subscriber: Rs 5,880

Option                Saved (subs)      Spend (Rs)    Net value (Rs)     ROI
----------------------------------------------------------------------------
Do nothing                     0.0               0                 0       -
Retention call                19.8          90,000            26,424    0.3x
Bill credit + call            33.0         165,000            29,040    0.2x

Best option by net value: Bill credit + call (net Rs 29,040)

--- Stress test: halve the most important assumption (reduction %) ---
Retention call       at half effectiveness (7.5% reduction): net Rs -31,788
Bill credit + call   at half effectiveness (12.5% reduction): net Rs -67,980
